# Phase 1: Support-set study on V2

## Section 1 - Purpose and overview

This study chooses the reference frames of V2, the support set. V2 classifies each patch by its nearest labelled reference patch (multi-prototype study, Section 8). With background margin 0 (margin study, Section 9) it is the current method of perception.

The support set of V2 has 10 frames, all chosen while the method was V0, one mean prototype per class. Eight come from the support-set study, where more frames helped V0 little: a frame aimed at a rare appearance brings only a few patches and barely moves the mean (support-set study, Sections 6.1 and 8). Two were added in the multi-prototype study by a coverage rule: for each piece, at least one example of each state in which it appears during the assembly (multi-prototype study, Section 1). Those two frames were motivated by errors seen on the validation set (multi-prototype study, Section 8). V2 keeps every labelled patch as a prototype, so a single frame with a missing appearance can change its predictions (Mensink et al. 2013, Sect. 1). The frames of the support set have never been chosen for V2.

Several limits of V2 are known from the previous studies. A loose arm is missed or confused near the hands. The hat is often missed on the table and in the hand. The mouth is recognized well only when its front is visible; seen from the side, its pin is predicted as nose. Hat patches appear in the cavity of the base when the base is alone. There are false positives on the watch, and on objects never seen.

An exploration suggests that the frames of the support set are the main lever. With the two frames of 1_disassembled_controlled_caos added to the support set, the loose arm on the table is seen in 44% of the frames of the assembly_2 video instead of 12%, and the hat in 67% instead of 54% (scratch_support_swap.ipynb). This result is optimistic: those two frames were validation frames, and the video was used for exploration. It is a hypothesis to test, not a result.

The study looks ahead to Phase 2. State tracking needs to know, for every piece, whether it is on the table, in a hand or mounted. A wrong pick is signalled before the error, a wrong mounting when it happens. Perception is therefore evaluated separately in three moments: pieces at rest on the table, the pick, and the mounting.

The criterion for the support set extends the coverage rule of the multi-prototype study: one frame for every state of every piece, seen as in use, without redundant frames. Seen as in use means from the headset, at working distance and with the framing of the assembly. The viewpoint of a piece is treated as an annotated attribute of each frame, as in PASCAL VOC (Everingham et al. 2010, Sect. 3.3). The mouth must be recognized in every state, from the moment it lies on the table.

The validation set changes. It now contains only frames of assembly_2: the 6 frames used so far and 26 new frames, chosen from the event log and the raw video by rules fixed before any result was seen (Section 3). The two frames of 1_disassembled_controlled_caos become candidates for the support set. The test recording, assembly_1, is not read, and is used once at the end of Phase 1. Splits are made by recording, as in Assembly101 (Sener et al. 2022, Sect. 4.6).

Choices made on the validation set make its results optimistic, more so with few frames and many variants (Cawley and Talbot 2010, Sect. 4.4 and 5). The support set is therefore chosen without the validation set, on the reference frames only (Section 6), and the validation set is used once, by a decision rule written before it is run (Section 8). The rule is added to this section before Section 8 is run.

The study is organized as follows. Section 2 sets the configuration. Section 3 builds and checks the new validation set. Section 4 checks that the current V2 reproduces the margin study. Section 5 lists the states of the pieces and the states covered by the current support set and the candidates. Section 6 chooses the support set, and Section 7 its background margin. Section 8 applies the decision rule. Section 9 looks at continuous perception on assembly_2, as an exploration. Section 10 gives the decision and the conclusions.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 1:

Mensink et al. 2013, Sect. 1: the nearest neighbour classifier keeps every training example, while a class mean summarizes them.

Everingham et al. 2010, Sect. 3.3: the viewpoint of each object is annotated as an attribute.

Sener et al. 2022, Assembly101, Sect. 4.6: training, validation and test splits are made by video, and the test ground truth is withheld.

Cawley and Talbot 2010, Sect. 4.4: selecting by optimizing a criterion on a finite sample over-fits, more with small samples and many choices. Sect. 5: a performance estimate on the data used for selection is optimistically biased.

Previous studies: support-set study (v1.3), Sections 6.1 and 8; multi-prototype study (v1.5), Sections 1 and 8; margin study (v1.6), Section 9; scratch_support_swap.ipynb (exploratory, not part of any study).

------------------------------------------------------------------------------------------------------------------------------------------

## Section 2 - Configuration

All paths, frames and hyperparameters in one place. The values are those of the margin study (Section 2): V2 on the same 10 reference frames, with the background margin decided there, 0. Three things change.

The validation set contains only assembly_2: the 6 frames used so far and the 26 new frames, 32 in all, each with its moment (table, pick or mount) and its piece. The rules that chose them are given and checked in Section 3. The frames are listed in time order, the order of the video.

The two frames of 1_disassembled_controlled_caos, validation frames until the margin study, become candidates for the support set (Section 1). Other candidates are added in Section 5, once the states not covered by the current support set are known.

The 8 validation frames of the margin study are kept as a separate list, used only to check in Section 4 that the current V2 reproduces that study.

In [1]:
# 2.1 Dependencies: install once per environment with pip install -r requirements.txt (README, Setup)

In [2]:
# 2.2 Imports
%load_ext autoreload
%autoreload 2

import sys
import json
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from scipy import ndimage

In [3]:
# 2.3 Paths and shared module
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))   # makes src/perception importable from notebooks/

MODEL_DIR = PROJECT_ROOT / "models" / "c_radio_weights"
FRAMES_DIR = PROJECT_ROOT / "data" / "extracted_frames"
VIDEO_PATH = PROJECT_ROOT / "data" / "raw_videos" / "assembly_2.mp4"                # Sections 3 and 9
EVENTS_PATH = PROJECT_ROOT / "configs" / "assembly_2_events.json"                   # Sections 3 and 9
MARGIN_STUDY_DIR = PROJECT_ROOT / "outputs" / "phase_1_margin_study"                # decision and labels of the margin study (Section 4)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase_1_support_set_v2_study"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"MODEL_DIR": MODEL_DIR, "FRAMES_DIR": FRAMES_DIR, "SRC_DIR": SRC_DIR,
                   "VIDEO_PATH": VIDEO_PATH, "EVENTS_PATH": EVENTS_PATH,
                   "margin study decision": MARGIN_STUDY_DIR / "decision.json"}.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status}] {name}: {path}")

# Decision of the margin study (Section 9), printed so that this notebook records what it starts from
print("\nMargin study decision:")
print(json.dumps(json.loads((MARGIN_STUDY_DIR / "decision.json").read_text()), indent=2))

# Shared code (src/perception)
from perception.classes import CLASS_NAMES, BACKGROUND_ID, HAND_LABEL, PIECE_LABELS, IGNORE_ID, PATCH_LABEL_NAMES, CLASS_COLORS
from perception.features import FeatureExtractor
from perception.labels import sequence_of, check_annotation, rasterize_annotation, patch_label_grid
from perception.prototypes import (patch_prototypes, sort_by_class, class_similarities_gpu, assign_labels, l2_normalize)
from perception.metrics import (confusion_matrix, iou_from_confusions, mean_iou, ignore_boundary_ring,
                                MERGED_NAMES, MERGED_PIECES, merge_arms, merged_ids,
                                frame_iou, frame_miou, bootstrap_miou, bootstrap_class_iou, percentile_interval)
from perception.visualization import label_grid_to_rgb, render_overlay

[OK] MODEL_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\models\c_radio_weights
[OK] FRAMES_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\data\extracted_frames
[OK] SRC_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\src
[OK] VIDEO_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\data\raw_videos\assembly_2.mp4
[OK] EVENTS_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\configs\assembly_2_events.json
[OK] margin study decision: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\outputs\phase_1_margin_study\decision.json

Margin study decision:
{
  "variant": "V2",
  "background_margin": 0.0,
  "decided_margin": {
    "V0": 0.15499374270439148,
    "V2": 0.0
  },
  "m_star_cross_validation": {
    "V0": 0.15499374270439148,
    "V2": 0.020618915557861328
  },
  "m_star_bootstrap_95": {
    "V0": [
      0.1531127691268921,
      0.18741735741496043
    ],
    "V2": [
      0.0030605196952819824,
      0.04309642314910889
    ]
  },
  "v2

In [4]:
# 2.4 Hyperparameters
# Same values as the margin study (Section 2.4), with the margin decided there for V2 (Section 9).

# --- Feature extraction ---
DEVICE = "cuda"
DTYPE = torch.bfloat16          # input resolution: native 1024x1024 (resolution study)

# --- Patch labelling ---
MIN_LABEL_COVERAGE = 0.5        # same rule as the pipeline (Section 5)

# --- Classification ---
MARGIN = 0.0                    # background margin of V2, decided in the margin study (Section 9): each patch takes its most similar class

# --- Evaluation ---
N_BOOTSTRAP = 1000              # resamples of the frames (support-set study, Section 5.4)
BOUNDARY_RING = 1               # secondary metric: background patches within 1 patch of an object are ignored

# --- Reproducibility ---
RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
_ = torch.manual_seed(RANDOM_SEED)

In [5]:
# 2.5 Frames and split
# Validation: assembly_2 only, used once, by the decision rule (Section 8).
# Test: held out until the end of Phase 1. This notebook never reads a test frame.
VALIDATION_SEQUENCES = {"assembly_2"}
TEST_SEQUENCES = {"assembly_1"}
assert not VALIDATION_SEQUENCES & TEST_SEQUENCES


def frame_path(frame, suffix=".jpg"):
    """Path of a frame or of its annotation. Refuses test frames, so the test set cannot be read by mistake."""
    assert sequence_of(frame) not in TEST_SEQUENCES, f"{frame} belongs to the held-out test set"
    return FRAMES_DIR / f"{frame}{suffix}"


# Current support set: the 10 frames of the multi-prototype and margin studies, in the same order (margin study, Section 2.5)
CURRENT_SUPPORT = [
    {"frame": "5_assembled_f0002",            "role": "assembled",                   "use_labels": PIECE_LABELS},
    {"frame": "caos_f0149",                   "role": "partially assembled",         "use_labels": PIECE_LABELS},
    {"frame": "5_assembled_f0019",            "role": "assembled, 3/4 view",         "use_labels": PIECE_LABELS},
    {"frame": "3_disassembled_ordered_f0001", "role": "disassembled",                "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "2_disassembled_caos_f0016",    "role": "disassembled, top-down view", "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "Take_3_1_Base_f0099",          "role": "base only, upright",          "use_labels": {"base"}},
    {"frame": "Take_3_1_Base_f0131",          "role": "base only, lying",            "use_labels": {"base"}},
    {"frame": "caos_f0211",                   "role": "hands",                       "use_labels": {HAND_LABEL}},
    {"frame": "4_partially_assembled_f0002",  "role": "body on base, face incomplete (empty mouth slot); mouth loose at 3/4",
     "use_labels": {"body", "mouth"}},
    {"frame": "caos_f0043",                   "role": "mouth loose, side view",      "use_labels": {"mouth"}},
]

# Candidates known before Section 5: the two frames of 1_disassembled_controlled_caos (Section 1).
# Base stays out of their labels, as it was excluded on these frames when they were validation frames (margin study, Section 2.5).
# Their final use_labels are set in Section 5, with the other candidates.
CANDIDATES = [
    {"frame": "1_disassembled_controlled_caos_f0001", "role": "disassembled, front view", "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "1_disassembled_controlled_caos_f0016", "role": "disassembled, side view",  "use_labels": PIECE_LABELS - {"base"}},
]

# Validation: 32 frames of assembly_2 in time order. moment = table, pick or mount (rules in Section 3).
# For pick and mount, piece is the piece being picked or mounted; for table, the state of the figure.
# old = one of the 6 frames used since the support-set study.
_VALIDATION = [
    (3,  "table", None,           "all pieces loose",                     True),
    (4,  "pick",  "base",         "base just picked",                     False),
    (5,  "mount", "base",         "base placed at the work position",     False),
    (8,  "table", None,           "base alone at the work position",      False),
    (10, "pick",  "body",         "body touched, hand mostly outside",    False),
    (12, "mount", "body",         "body put on the base",                 False),
    (17, "table", None,           "body on base, mouth not yet picked",   True),
    (18, "pick",  "mouth",        "mouth touched",                        False),
    (19, "mount", "mouth",        "pin of the mouth at the slot",         False),
    (24, "table", None,           "mouth mounted",                        False),
    (26, "pick",  "moustache",    "moustache picked",                     False),
    (28, "pick",  "nose",         "nose touched, moustache in the other hand", False),
    (30, "mount", "nose",         "nose threaded into the moustache, in hand", True),
    (33, "mount", "nose+moustache", "nose and moustache entering the slot", False),
    (37, "table", None,           "nose and moustache mounted",           False),
    (42, "pick",  "eyes",         "hand on the eyes",                     False),
    (43, "mount", "eyes",         "eyes entering the slot",               False),
    (46, "table", None,           "face complete, ears loose",            True),
    (50, "pick",  "ear",          "ear (image right) in hand",            False),
    (51, "mount", "ear",          "ear (image right) on the head",        False),
    (52, "table", None,           "one ear mounted, hands in view",       False),
    (53, "pick",  "ear",          "ear (image left) in hand",             False),
    (54, "mount", "ear",          "ear (image left) being pushed in",     False),
    (57, "table", None,           "ears mounted, arms loose",             True),
    (59, "pick",  "arm_1",        "arm_1 just picked, far from the body", False),
    (60, "mount", "arm_1",        "arm_1 touching its hole",              False),
    (63, "pick",  "arm_2",        "arm_2 just picked, far from the body", False),
    (65, "mount", "arm_2",        "arm_2 in its hole, hand on it",        False),
    (68, "table", None,           "arms mounted, hat loose",              True),
    (70, "pick",  "hat",          "hat in hand",                          False),
    (73, "mount", "hat",          "hat entering the head",                False),
    (77, "table", None,           "figure complete, hands still",         False),
]
VALIDATION_FRAMES = [{"frame": f"assembly_2_f{index:04d}", "index": index, "moment": moment, "piece": piece,
                      "note": note, "old": old} for index, moment, piece, note, old in _VALIDATION]
EXCLUDED = [set() for _ in VALIDATION_FRAMES]   # no class is out of definition on assembly_2 frames

# Validation of the margin study (Section 2.5): used only for the regression check of Section 4
MARGIN_STUDY_VALIDATION = [
    {"frame": "assembly_2_f0003"}, {"frame": "assembly_2_f0017"}, {"frame": "assembly_2_f0030"},
    {"frame": "assembly_2_f0046"}, {"frame": "assembly_2_f0057"}, {"frame": "assembly_2_f0068"},
    {"frame": "1_disassembled_controlled_caos_f0001", "exclude": {"base"}},
    {"frame": "1_disassembled_controlled_caos_f0016", "exclude": {"base"}},
]

# Checks: no leakage, every file present, the planned composition
for item in CURRENT_SUPPORT + CANDIDATES:
    assert sequence_of(item["frame"]) not in VALIDATION_SEQUENCES | TEST_SEQUENCES, f"Leakage: {item['frame']}"
    item["image_path"], item["json_path"] = frame_path(item["frame"]), frame_path(item["frame"], ".json")
    assert item["image_path"].exists() and item["json_path"].exists(), f"Missing files: {item['frame']}"
for val in VALIDATION_FRAMES:
    assert sequence_of(val["frame"]) in VALIDATION_SEQUENCES, f"{val['frame']} is not a validation frame"
    val["image_path"], val["json_path"] = frame_path(val["frame"]), frame_path(val["frame"], ".json")
    assert val["image_path"].exists() and val["json_path"].exists(), f"Missing files: {val['frame']}"
indices = [val["index"] for val in VALIDATION_FRAMES]
assert indices == sorted(set(indices)), "Validation frames must be unique and in time order"
assert sum(val["old"] for val in VALIDATION_FRAMES) == 6
assert {v["frame"] for v in MARGIN_STUDY_VALIDATION[:6]} == {v["frame"] for v in VALIDATION_FRAMES if v["old"]}

GT_LABELS = PIECE_LABELS | {HAND_LABEL}   # validation ground truth: every official class; legacy labels become ignored

composition = pd.crosstab(pd.Series([v["moment"] for v in VALIDATION_FRAMES], name="moment"),
                          pd.Series(["old" if v["old"] else "new" for v in VALIDATION_FRAMES], name="frames"),
                          margins=True, margins_name="total")
print(f"{len(CURRENT_SUPPORT)} reference frames, {len(CANDIDATES)} candidates so far, {len(VALIDATION_FRAMES)} validation frames")
display(composition)
display(pd.DataFrame(VALIDATION_FRAMES).set_index("frame")[["moment", "piece", "note", "old"]])

10 reference frames, 2 candidates so far, 32 validation frames


frames,new,old,total
moment,,,
mount,10,1,11
pick,11,0,11
table,5,5,10
total,26,6,32


,moment,piece,note,old
frame,,,,
assembly_2_f0003,table,None,all pieces loose,True
assembly_2_f0004,pick,base,base just picked,False
assembly_2_f0005,mount,base,base placed at the work position,False
assembly_2_f0008,table,None,base alone at the work position,False
assembly_2_f0010,pick,body,"body touched, hand mostly outside",False
assembly_2_f0012,mount,body,body put on the base,False
assembly_2_f0017,table,None,"body on base, mouth not yet picked",True
assembly_2_f0018,pick,mouth,mouth touched,False
assembly_2_f0019,mount,mouth,pin of the mouth at the slot,False


In [6]:
# 2.6 Annotation fingerprints: data/ is not tracked by git, so the notebook records which annotations it used.
# Fingerprints recorded by the margin study (Section 2.6), to check that the files it used are unchanged.
MARGIN_STUDY_SHA = {
    "5_assembled_f0002": "e85d1c179582", "caos_f0149": "14f88d7cf080", "5_assembled_f0019": "1cd8cdcc2c9b",
    "3_disassembled_ordered_f0001": "319c3601f0a1", "2_disassembled_caos_f0016": "f17ab6af8218",
    "Take_3_1_Base_f0099": "853e60024c17", "Take_3_1_Base_f0131": "f07f67aae3a9", "caos_f0211": "3c56b4804e54",
    "4_partially_assembled_f0002": "32a94339f2b4", "caos_f0043": "3b555dd93c2c",
    "assembly_2_f0003": "350cee9c49e9", "assembly_2_f0017": "ce242dfd4544", "assembly_2_f0030": "69a762c4d37d",
    "assembly_2_f0046": "63664e7faa10", "assembly_2_f0057": "7bf765f2fda9", "assembly_2_f0068": "326615769d7c",
    "1_disassembled_controlled_caos_f0001": "6ba5febd2141", "1_disassembled_controlled_caos_f0016": "1d3db54d65ad",
}


def fingerprint(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]


rows = ([(item["frame"], "current support", item["json_path"]) for item in CURRENT_SUPPORT]
        + [(item["frame"], "candidate", item["json_path"]) for item in CANDIDATES]
        + [(val["frame"], f"validation ({val['moment']})", val["json_path"]) for val in VALIDATION_FRAMES])
fingerprints = pd.DataFrame([{"frame": frame, "set": role, "annotation sha256": fingerprint(path),
                              "margin study": ("new" if frame not in MARGIN_STUDY_SHA
                                               else "same" if fingerprint(path) == MARGIN_STUDY_SHA[frame] else "CHANGED")}
                             for frame, role, path in rows]).set_index("frame")
display(fingerprints)
print(fingerprints["margin study"].value_counts().to_string())

,set,annotation sha256,margin study
frame,,,
5_assembled_f0002,current support,e85d1c179582,same
caos_f0149,current support,14f88d7cf080,same
5_assembled_f0019,current support,1cd8cdcc2c9b,same
3_disassembled_ordered_f0001,current support,319c3601f0a1,same
2_disassembled_caos_f0016,current support,f17ab6af8218,same
Take_3_1_Base_f0099,current support,853e60024c17,same
Take_3_1_Base_f0131,current support,f07f67aae3a9,same
caos_f0211,current support,3c56b4804e54,same
4_partially_assembled_f0002,current support,32a94339f2b4,same


margin study
new     26
same    18


In [7]:
# 2.7 Environment
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

Python       : 3.10.0
PyTorch      : 2.5.1+cu121
Transformers : 5.17.0
CUDA         : True
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU


------------------------------------------------------------------------------------------------------------------------------------------

References Section 2: none. The values and the current support set are those of the margin study (Sections 2.4, 2.5 and 9); the validation frames are chosen by the rules of Section 3.

------------------------------------------------------------------------------------------------------------------------------------------